Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs/001/L_Fore/01.bmp'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)


Preprocessing for Training:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"
IMAGE_SIZE = (128, 60)
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
TRAIN_SAMPLES_PER_SUBJECT = 5

train_data = []
train_labels = []

# === FUNCTION TO FUSE FIRST 7 IMAGES OF EACH FINGER ===
def load_and_fuse_finger_images(subject_path, subject_id):
    fused_samples = []
    labels = []
    for i in range(1, 6):  # Only 5 images for Protocol 2
        fused_vector = []
        complete = True
        print(f"\n📁 Subject {subject_id} - Image Index {i:02d}")

        for finger in FINGER_LIST:
            img_path = os.path.join(subject_path, finger, f"{i:02d}.bmp")
            print(f"🖼️ Loading: {img_path}")

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"⚠️ Missing image: {img_path}")
                complete = False
                break

            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / np.std(img_eq)
            fused_vector.append(img_norm.flatten())

        if complete:
            fused = np.concatenate(fused_vector)
            fused_samples.append(fused)
            label = f"{subject_id}_fused_img{i:02d}"
            labels.append(label)
            print(f"✅ Fused sample created and labeled: {label}")
        else:
            print(f"❌ Incomplete fusion for Subject {subject_id}, Image {i:02d}")

    return fused_samples, labels

# === LOAD TRAINING DATA FOR ALL SUBJECTS ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="Loading Protocol 1 - Train Data"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue
    fused_images, fused_labels = load_and_fuse_finger_images(subject_path, subj)
    train_data.extend(fused_images)
    train_labels.extend(fused_labels)

# === CONVERT TO NUMPY ARRAYS ===
train_data = np.array(train_data)
train_labels = np.array(train_labels)

# === SHAPE CHECK ===
print("\n✅ Train Data Shape:", train_data.shape)
print("✅ Train Labels Shape:", train_labels.shape)
print("✅ Train Labels Example:", train_labels[:5])


Test:

In [ ]:
import numpy as np

# Step 1: Center the training data
mean_vector = np.mean(train_data, axis=0)
centered_data = train_data - mean_vector  # Shape: (n_samples, n_features)

# Step 2: Compute subject-to-subject Gram matrix
gram_matrix = centered_data @ centered_data.T  # Shape: (n_samples, n_samples)

# Step 3: Eigen decomposition of Gram matrix
eig_vals, eig_vecs = np.linalg.eigh(gram_matrix)  # ascending order

# Step 4: Sort eigenvalues/vectors in descending order
sorted_indices = np.argsort(-eig_vals)
eig_vals = eig_vals[sorted_indices]
eig_vecs = eig_vecs[:, sorted_indices]

# Step 5: Filter valid eigenvectors (non-zero eigenvalues)
valid_indices = eig_vals > 1e-10
eig_vals_valid = eig_vals[valid_indices]
eig_vecs_valid = eig_vecs[:, valid_indices]

# ✅ Step 6: Project all valid eigenvectors back to original feature space
eig_vecs_full = (centered_data.T @ eig_vecs_valid) / np.sqrt(eig_vals_valid)

# Step 7: Project training data into full PCA space
train_data_pca = centered_data @ eig_vecs_full  # Shape: (n_samples, k)

# Final output
print("✅ PCA-transformed training data shape:", train_data_pca.shape)
print("✅ Number of principal components used:", eig_vecs_full.shape[1])


Preprocessing for Testing:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"  # Replace with your actual dataset path
IMAGE_SIZE = (128, 60)
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
TEST_INDICES = [5, 6, 7, 8, 9, 10]  # Protocol 2 test images

test_data = []
test_labels = []

# === LOAD TEST DATA FOR ALL SUBJECTS ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="Loading Protocol 1 - Test Data"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for img_idx in TEST_INDICES:
        fused_vector = []
        complete = True
        print(f"\n📁 Subject {subj} - Image Index {img_idx:02d}")

        for finger in FINGER_LIST:
            img_path = os.path.join(subject_path, finger, f"{img_idx:02d}.bmp")
            print(f"🖼️ Loading: {img_path}")

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"⚠️ Missing image: {img_path}")
                complete = False
                break

            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
            fused_vector.append(img_norm.flatten())

        if complete:
            sample_vector = np.concatenate(fused_vector)
            test_data.append(sample_vector)
            label = f"{subj}_fused_img{img_idx:02d}"
            test_labels.append(label)
            print(f"✅ Fused test sample created and labeled: {label}")
        else:
            print(f"❌ Incomplete fusion for Subject {subj}, Image {img_idx:02d}")

# === CONVERT TO NUMPY ARRAYS ===
test_data = np.array(test_data)
test_labels = np.array(test_labels)

# === SHAPE CHECK ===
print("\n✅ Test data loaded successfully.")
print(f"🧪 Total test samples: {len(test_data)}")
if len(test_data) > 0:
    print(f"🧾 Example vector shape: {test_data[0].shape}")
    print(f"🧾 Sample label: {test_labels[0]}")


Benchmarking

In [ ]:
correct_matches = 0
total_tests = len(test_data)

# 📉 Project test data into PCA space
centered_test_data = test_data - mean_vector
proj_test_data = centered_test_data @ eig_vecs_full

# 🔍 Loop through each test sample
for i in range(total_tests):
    proj_test = proj_test_data[i]
    true_label = test_labels[i]          # e.g., "001_fused_img08"

    # 📏 Manhattan distance to all training vectors
    distances = np.sum(np.abs(train_data_pca - proj_test), axis=1)

    # 🏆 Find the closest match
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]  # e.g., "001_fused_img04"

    # 🎯 Compare only subject IDs
    true_id = true_label.split('_')[0]
    pred_id = predicted_label.split('_')[0]

    # ✅ ID match check
    if pred_id == true_id:
        match_result = "✅"
        correct_matches += 1
    else:
        match_result = "❌"

    print(f"Test sample {i+1}: Predicted = {predicted_label}, Actual = {true_label} {match_result}")

# 📈 Final accuracy output
accuracy = (correct_matches / total_tests) * 100
print(f"\n🎉 Person Identification Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests} correct matches)")